In [0]:
# Project         : Procurement Analytics using Databricks & Power BI
# Layer           : Silver
# Notebook        : Silver_Purchase_Requisitions
# Source          : Purchase_Requisitions.csv
# Target          : procurement.silver.silver_Purchase_Requisitions
#
# Author          : V R Mutyala
# Created Date    : 21-Jul-2026
# Last Modified   : 21-Jul-2026
#
# Description
# -----------
# This notebook loads Cleaned Purchase_Requisitions master data into the Silver layer.
# It validates the source data, separates duplicate records, adds audit columns, and stores the results as Delta tables.

# ==============================================================================
# Business Objective
# ==============================================================================
#
# Read employee data from the Bronze layer, apply data cleansing,standardization, and business rules to create a trusted Silver Delta table.
# ==============================================================================

In [0]:
%run ../01_Config/Config

In [0]:
%run ../05_Helper_Functions/Helper_functions

In [0]:
# Import Libraries and Widgets
from pyspark.sql import DataFrame
from pyspark.sql.functions import (col, lit,current_timestamp,when,count,trim,coalesce,initcap,lower)
from pyspark.sql.types import (StructType, StructField, StringType,IntegerType,DoubleType,DecimalType)
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
from pyspark.sql.functions import try_to_timestamp, to_date, try_to_date

In [0]:
# ============================================================
# Read Bronze Invoice Table
# ============================================================

bronze_purchase_requisitions_df = read_delta(BRONZE_PURCHASE_REQUISITIONS)

preview(bronze_purchase_requisitions_df,"Bronze purchase_requisitions")

In [0]:
#============================================
# Create Sliver DataFrame
#===========================================
silver_purchase_requisitions_df = bronze_purchase_requisitions_df

In [0]:
# ============================================================
# Apply business transformations
# ============================================================
silver_purchase_requisitions_df = (silver_purchase_requisitions_df

    # Trim string columns
    .withColumn("pr_id", trim(col("pr_id")))
    .withColumn("employee_id", trim(col("employee_id")))
    .withColumn("department_id", trim(col("department_id")))
    .withColumn("product_id", trim(col("product_id")))
    .withColumn("priority", trim(col("priority")))
    .withColumn("status", initcap(trim(col("status"))))

    # Date columns
    .withColumn("pr_date",
                to_date(trim(col("pr_date")), "yyyy-MM-dd"))
    .withColumn("approval_date",
                to_date(trim(col("approval_date")), "yyyy-MM-dd"))

    # Numeric columns (no trim)
    .withColumn("quantity_requested", col("quantity_requested").cast("int"))
    .withColumn("estimated_unit_price", col("estimated_unit_price").cast("decimal(10,2)"))
    .withColumn("estimated_total", col("estimated_total").cast("decimal(12,2)"))

    # Standardize approver_id
    .withColumn(
        "approver_id",
        when(
            upper(trim(col("approver_id"))).isin("E0079", "E00 79"),
            "EOO 79"
        )
        .when(
            trim(col("approver_id")) == "",
            None
        )
        .otherwise(upper(trim(col("approver_id"))))
    )
)



In [0]:
# ============================================================
# Identify Invalid Purchase_Requisitions Records
# NULL & Blank: pr_id
# NULL: pr_date, employee_id, department_id,product_id,
#       ,priority,status,approver_id,approval_date
# Zero: quantity_requested,estimated_unit_price,estimated_total
# ============================================================

invalid_purchase_requisitions = (
    silver_purchase_requisitions_df.filter(

        # PR ID
        col("pr_id").isNull()
        | (trim(col("pr_id")) == "")

        # PR Date
        | col("pr_date").isNull()

        # Employee
        | col("employee_id").isNull()
        | (trim(col("employee_id")) == "")

        # Department
        | col("department_id").isNull()
        | (trim(col("department_id")) == "")

        # Product
        | col("product_id").isNull()
        | (trim(col("product_id")) == "")

        # Quantity
        | col("quantity_requested").isNull()
        | (col("quantity_requested") <= 0)

        # Unit Price
        | col("estimated_unit_price").isNull()
        | (col("estimated_unit_price") <= 0)

        # Total
        | col("estimated_total").isNull()
        | (col("estimated_total") <= 0)

        # Priority
        | col("priority").isNull()
        | (trim(col("priority")) == "")

        # Status
        | col("status").isNull()
        | (trim(col("status")) == "")

        # Approved PR must have approval date
        | (
            (upper(trim(col("status"))) == "APPROVED")
            & col("approval_date").isNull()
        )
    )
)

print(
    f"Invalid Purchase Requisition Records: "
    f"{invalid_purchase_requisitions.count()}"
)

display(invalid_purchase_requisitions)

In [0]:
# ============================================================
# Add Audit Metadata
# ============================================================

invalid_purchase_requisitions = (invalid_purchase_requisitions.withColumn("audit_timestamp",current_timestamp())
    .withColumn("source_table",lit("purchase_requisitions"))
    .withColumn("pipeline_layer",lit("Silver"))
    .withColumn("issue_type",lit("Invalid Record")))

display(invalid_purchase_requisitions)

In [0]:
# ============================================================
# Write Invalid PURCHASE_REQUISITIONS to Audit Table
# ============================================================

if invalid_purchase_requisitions.count() > 0:
    write_delta(invalid_purchase_requisitions,AUDIT_INVALID_PURCHASE_REQUISITIONS,mode="overwrite")
    print("Invalid invoice records written.")
else:
    print("No invalid invoice records found.")

In [0]:
# ============================================================
# Remove Invalid Records
# ============================================================

# ============================================================
# Remove Invalid Purchase Requisition Records
# ============================================================

silver_purchase_requisitions_df = silver_purchase_requisitions_df.filter(

    # PR ID
    col("pr_id").isNotNull() &
    (trim(col("pr_id")) != "") &

    # PR Date
    col("pr_date").isNotNull() &

    # Employee ID
    col("employee_id").isNotNull() &
    (trim(col("employee_id")) != "") &

    # Department ID
    col("department_id").isNotNull() &
    (trim(col("department_id")) != "") &

    # Product ID
    col("product_id").isNotNull() &
    (trim(col("product_id")) != "") &

    # Quantity Requested
    col("quantity_requested").isNotNull() &
    (col("quantity_requested") > 0) &

    # Estimated Unit Price
    col("estimated_unit_price").isNotNull() &
    (col("estimated_unit_price") > 0) &

    # Estimated Total
    col("estimated_total").isNotNull() &
    (col("estimated_total") > 0) &

    # Priority
    col("priority").isNotNull() &
    (trim(col("priority")) != "") &

    # Status
    col("status").isNotNull() &
    (trim(col("status")) != "") &

    # Approved PR must have Approver ID
    (
        (upper(trim(col("status"))) != "APPROVED") |
        col("approver_id").isNotNull()
    ) &

    # Approved PR must have Approval Date
    (
        (upper(trim(col("status"))) != "APPROVED") |
        col("approval_date").isNotNull()
    )
)

display(silver_purchase_requisitions_df)

In [0]:
# ============================================================
# Remove Duplicate PURCHASE_REQUISITIONS
# ============================================================

window_spec = Window.partitionBy("pr_id").orderBy("pr_date")

silver_purchase_requisitions_df = (silver_purchase_requisitions_df.withColumn("row_num",row_number().over(window_spec))
                      .filter(col("row_num") == 1).drop("row_num"))
preview(silver_purchase_requisitions_df,"Silver Purchase_Requisitions")


In [0]:
# ============================================================
# Write Silver Delta Table
# ============================================================

silver_purchase_requisitions_df = (silver_purchase_requisitions_df
                         .withColumn("silver_load_timestamp", current_timestamp())
                         .withColumn("pipeline_layer", lit("Silver"))
)

In [0]:
# ============================================================
# Write Silver Delta Table
# ============================================================

write_delta(df=silver_purchase_requisitions_df,table_name=SILVER_PURCHASE_REQUISITIONS)

In [0]:
# ============================================================
# Validate Summary
# ============================================================

bronze_count = bronze_purchase_requisitions_df.count()
invalid_count = invalid_purchase_requisitions.count()
silver_count = silver_purchase_requisitions_df.count()

duplicate_removed = bronze_count - invalid_count - silver_count

print("=" * 60)
print("Silver purchase_requisitions Load Completed Successfully")
print("=" * 60)

print(f"Bronze Records            : {bronze_count}")
print(f"Invalid Records Removed   : {invalid_count}")
print(f"Duplicate Records Removed : {duplicate_removed}")
print(f"Silver Records            : {silver_count}")